# CSPF — post-fix analysis (appendix tables, cluster-aware CIs, paired tests)
Input: `ALL_MODELS_combined_FIXED.csv` written by `CSPF_fix_alignment.ipynb`. **No GPU needed, no model is run.**
Outputs (all small CSVs, saved to Drive and zipped): per-dataset×model accuracy/F1 (Table A2), confusion counts (Fig. A2/A3), per-dataset summary (Table A3), review-level cluster bootstrap CIs, Cochran's Q + pairwise McNemar-Holm, the restricted subsets (propositional families, Φ ≥ 0.75), family×position table.
Run all cells in order; set the paths in cell 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
FIXED_CSV = "/content/drive/MyDrive/postdoc-papers/paper2/ALL_MODELS_combined_FIXED.csv"   # <- same as in the fix notebook
OUT_DIR   = "/content/drive/MyDrive/postdoc-papers/paper2/results_FIXED/post"              # results are written here (Drive)
N_BOOT    = 1000
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd, numpy as np, os, zipfile, itertools, warnings
warnings.filterwarnings("ignore")
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
BASE8  = ["ALBERT","BERT","DistilBERT","ELECTRA","GPT2","RoBERTa","T5","XLNet"]
print(len(MODELS), "models:", MODELS)
base = ["source_dataset","review_idx","sentiment","perturbation_family","severity","position_strategy","linguistic_target","sem_preservation_score"]
use = base + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
df = pd.read_csv(FIXED_CSV, usecols=use, low_memory=False)
df["key"] = df.source_dataset.astype(str) + "::" + df.review_idx.astype(str)
df["y"] = (df.sentiment=="positive").astype(np.int8)
for m in MODELS:
    for s in ("orig","pert"):
        df[f"{m}_{s}"] = (df[f"{m}_{s}_label"]=="positive").astype(np.int8); del df[f"{m}_{s}_label"]
    df[f"{m}_flip"] = (df[f"{m}_orig"]!=df[f"{m}_pert"]).astype(np.int8)
codes, uniq = pd.factorize(df.key); df["cl"] = codes
print(len(df), "pairs |", len(uniq), "clusters (source reviews)")

In [ ]:
# ── Table A2: dataset x model accuracy / macro-F1, original (unique reviews) vs perturbed (all pairs) ──
def f1_macro(y, p):
    f=[]
    for c in (0,1):
        tp=((p==c)&(y==c)).sum(); fp=((p==c)&(y!=c)).sum(); fn=((p!=c)&(y==c)).sum()
        f.append(0.0 if tp==0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))
u = df.drop_duplicates("key")
rows=[]
for ds, gu in u.groupby("source_dataset"):
    gp = df[df.source_dataset==ds]
    for m in MODELS:
        ao=(gu[f"{m}_orig"]==gu.y).mean(); ap=(gp[f"{m}_pert"]==gp.y).mean()
        fo=f1_macro(gu.y.values,gu[f"{m}_orig"].values); fp_=f1_macro(gp.y.values,gp[f"{m}_pert"].values)
        rows.append(dict(Model=m,Dataset=ds,Original_Accuracy=ao,Original_F1=fo,Perturbed_Accuracy=ap,Perturbed_F1=fp_,
                         Accuracy_Drop=ao-ap,F1_Drop=fo-fp_,n_reviews=len(gu),n_pairs=len(gp)))
A2=pd.DataFrame(rows).round(4); A2.to_csv(f"{OUT_DIR}/tableA2_dataset_model_performance.csv",index=False); print(A2.shape)

In [ ]:
# ── confusion counts (positive class = 'positive'): originals on unique reviews, perturbed on all pairs ──
rows=[]
for m in MODELS:
    for kind,d_,col in (("original",u,f"{m}_orig"),("perturbed",df,f"{m}_pert")):
        y=d_.y.values; p=d_[col].values
        rows.append(dict(model=m,text=kind,TP=int(((p==1)&(y==1)).sum()),TN=int(((p==0)&(y==0)).sum()),FP=int(((p==1)&(y==0)).sum()),FN=int(((p==0)&(y==1)).sum())))
CM=pd.DataFrame(rows); CM.to_csv(f"{OUT_DIR}/confusion_counts.csv",index=False); print(CM.head(4))
# ── per-dataset summary (Appendix A.5): mean over 16 and over the 8 base models ──
g=[]
for ds, gu in u.groupby("source_dataset"):
    gp=df[df.source_dataset==ds]
    r=dict(Dataset=ds,Pairs=len(gp),Reviews=len(gu))
    for tag,ms in (("all16",MODELS),("base8",[m for m in BASE8 if m in MODELS])):
        r[f"acc_orig_{tag}"]=np.mean([(gu[f"{m}_orig"]==gu.y).mean() for m in ms])
        r[f"acc_pert_{tag}"]=np.mean([(gp[f"{m}_pert"]==gp.y).mean() for m in ms])
        r[f"DVI_{tag}"]=np.mean([gp[f"{m}_flip"].mean() for m in ms])
    g.append(r)
DS=pd.DataFrame(g).round(4).sort_values("acc_orig_all16"); DS["gain_all16"]=(DS.acc_pert_all16-DS.acc_orig_all16).round(4)
DS.to_csv(f"{OUT_DIR}/tableA3_dataset_summary.csv",index=False); print(DS.to_string(index=False))
print("corr(DVI, acc_orig) all16:", round(np.corrcoef(DS.DVI_all16,DS.acc_orig_all16)[0,1],3), " base8:", round(np.corrcoef(DS.DVI_base8,DS.acc_orig_base8)[0,1],3))

In [ ]:
# ── review-level CLUSTER bootstrap (resample source reviews, not pairs) ──
rng=np.random.default_rng(42); K=len(uniq); FAMS=sorted(df.perturbation_family.unique()); SEV=["mild","medium","strong"]
flipM=df[[f"{m}_flip" for m in MODELS]].values.astype(np.float32)
def cluster_counts(mask):
    n=np.bincount(df.cl.values[mask],minlength=K).astype(np.float64)
    f=np.stack([np.bincount(df.cl.values[mask],weights=flipM[mask,j],minlength=K) for j in range(len(MODELS))],axis=1)
    return n,f
def boot(n,f,B=N_BOOT):
    out=np.empty((B,f.shape[1]))
    for b in range(B):
        w=np.bincount(rng.integers(0,K,K),minlength=K).astype(np.float64)
        out[b]=(w@f)/max(w@n,1)
    return out
rows=[]
allm=np.ones(len(df),bool)
def add(label,mask):
    n,f=cluster_counts(mask); est=f.sum(0)/n.sum(); bs=boot(n,f)
    for j,m in enumerate(MODELS):
        rows.append(dict(subset=label,model=m,FR=est[j],lo=np.percentile(bs[:,j],2.5),hi=np.percentile(bs[:,j],97.5),n_pairs=int(n.sum())))
    pooled=bs.mean(1); pe=f.sum()/(n.sum()*len(MODELS))
    rows.append(dict(subset=label,model="POOLED(16)",FR=pe,lo=np.percentile(pooled,2.5),hi=np.percentile(pooled,97.5),n_pairs=int(n.sum())))
add("ALL",allm)
PROP=["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]; MOD=["P03_negation","P04_sarcasm","P05_dilution","P07_contrastive","P09_intensity"]
add("propositional_P2_P8_P10_P11_P12",df.perturbation_family.isin(PROP).values)
add("modulating_P3_P4_P5_P7_P9",df.perturbation_family.isin(MOD).values)
add("Phi>=0.75",(df.sem_preservation_score>=0.75).values)
for fam_ in FAMS: add(fam_,(df.perturbation_family==fam_).values)
for fam_ in FAMS:
    for s in SEV:
        mk=((df.perturbation_family==fam_)&(df.severity==s)).values
        if mk.sum()>=300: add(f"{fam_}|{s}",mk)
CB=pd.DataFrame(rows).round(5); CB.to_csv(f"{OUT_DIR}/cluster_bootstrap_FR.csv",index=False)
print(CB[CB.subset=="ALL"][["model","FR","lo","hi"]].to_string(index=False))
print(CB[(CB.model=="POOLED(16)")&(~CB.subset.str.contains("[|]"))][["subset","FR","lo","hi","n_pairs"]].to_string(index=False))

In [ ]:
# ── paired analysis on the same pairs: Cochran's Q over the 16 models + pairwise McNemar with Holm ──
from scipy import stats
from statsmodels.stats.multitest import multipletests
F=flipM.astype(np.float64); k=F.shape[1]; Cj=F.sum(0); Ri=F.sum(1); T=Cj.sum()
Q=(k-1)*(k*(Cj**2).sum()-T**2)/(k*T-(Ri**2).sum()); print("Cochran Q =",round(Q,1),"df =",k-1,"p =",stats.chi2.sf(Q,k-1))
Bm=F.T@(1-F)          # b[i,j] = flips by i and not by j
rows=[]
for i,j in itertools.combinations(range(k),2):
    b,c=Bm[i,j],Bm[j,i]; chi=(abs(b-c)-1)**2/(b+c) if b+c>0 else 0
    rows.append(dict(model_i=MODELS[i],model_j=MODELS[j],only_i_flips=int(b),only_j_flips=int(c),chi2=chi,p_raw=stats.chi2.sf(chi,1)))
MC=pd.DataFrame(rows); MC["p_holm"]=multipletests(MC.p_raw,method="holm")[1]; MC["sig_holm_0.05"]=MC.p_holm<0.05
MC.to_csv(f"{OUT_DIR}/mcnemar_holm_pairs.csv",index=False); print("pairs significant after Holm:",int(MC["sig_holm_0.05"].sum()),"of",len(MC))
pd.DataFrame([dict(Q=Q,df=k-1,p=stats.chi2.sf(Q,k-1),n_items=len(df))]).to_csv(f"{OUT_DIR}/cochran_q.csv",index=False)

In [ ]:
# ── family x position (pooled over models) to separate position from family ──
fp=df.groupby(["perturbation_family","position_strategy"]).agg(n=("cl","size"),FR=("cl",lambda s: 0.0)).reset_index()
fp["FR"]=[flipM[((df.perturbation_family==a)&(df.position_strategy==b)).values].mean() for a,b in zip(fp.perturbation_family,fp.position_strategy)]
fp.round(5).to_csv(f"{OUT_DIR}/family_by_position_FR.csv",index=False); print(fp.round(4).to_string(index=False))

In [ ]:
# ── per-model x family x severity FR on the fixed data (cross-check of tables 01) + bundle ──
import glob
z=f"{OUT_DIR}/post_tables.zip"
with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(f"{OUT_DIR}/*.csv"): zf.write(f,os.path.basename(f))
print(os.path.getsize(z)/1e3,"KB ->",z)
try:
    from google.colab import files; files.download(z)
except Exception as e: print(e)